<a href="https://colab.research.google.com/github/NoeliaOrsini/jurisMind_AI_MultiAgente/blob/main/JurisMind_AI_MultiAgente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -U langchain
%pip install -U langgraph
%pip install -U langchain-community
%pip install arxiv
%pip install gradio
%pip install langchain-google-genai
%pip install -qU langchain-google-genai



In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

# Conecto las API Keys limpiando espacios
os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY").strip()
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY").strip()

# Configuro el cerebro
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)


In [ ]:
import arxiv
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool

# 1. Herramienta Web (Tavily)
# Agrego la descripción para que el agente sepa que debe priorizar Argentina y España
busca_web = TavilySearchResults(
    max_results=2,
    search_depth="advanced",
    max_tokens=600
)
busca_web.name = "busca_web"
busca_web.description = "Busca información legal, normativa y jurisprudencia actualizada. PRIORIDAD: Fuentes de Argentina y España."

# 2. Herramienta Académica (arXiv)
@tool
def tool_cientifica(query: str) -> str:
    """Busca papers académicos y doctrina científica sobre ética, inteligencia artificial y derecho en arXiv.
       PRIORIDAD: Utilizar para obtener marcos teóricos y doctrina legal enfocada en Argentina y España."""
    client = arxiv.Client()
    search = arxiv.Search(query=query, max_results=2)

    resultados = []
    for result in client.results(search):
        resultados.append(f"Título: {result.title}\nAutores: {[a.name for a in result.authors]}\nResumen: {result.summary}\n")

    return "\n---\n".join(resultados) if resultados else "No se encontraron papers académicos."

tools = [busca_web, tool_cientifica]

In [ ]:
from typing import TypedDict
from langgraph.prebuilt import create_react_agent

# 1. DEFINICIÓN DEL ESTADO
class AgentState(TypedDict):
    user_query: str
    web_answer: str
    scientific_answer: str
    router_decision: str
    final_answer: str

# 2. DEFINICIÓN DE LOS AGENTES
prompt_legal_web = """
Actúas como un abogado especialista en Derecho Digital.
FECHA ACTUAL: 2026.
Tu objetivo es investigar el impacto práctico, riesgos normativos o jurisprudencia en la web.
JURISDICCIÓN OBLIGATORIA: Argentina y España.
Utiliza la herramienta 'busca_web' para fundamentar tu respuesta.

REGLA DE ORO: Debes incluir una sección final llamada '### Referencias Web'
que liste los enlaces (URLs) específicos que utilizaste para este análisis.
"""
agente_web = create_react_agent(model=llm, tools=[busca_web], prompt=prompt_legal_web)

prompt_etica_doctrina = """
Actúas como un asesor experto en Ética de la IA, Gobernanza de Datos, Compliance y mitigación de riesgos técnicos.
FECHA ACTUAL: 2026.
Tu objetivo es realizar un análisis doctrinario.
JURISDICCIÓN OBLIGATORIA: Argentina y España.
Utiliza la herramienta 'tool_cientifica' de arXiv. Si buscas temas de Compliance, Ética o Privacidad, busca doctrina jurídica o técnica.
Extrae los títulos y autores clave que fundamenten el análisis.
REGLA DE ORO (CITACIÓN OBLIGATORIA):
Debes incluir al final una sección llamada '### Bibliografía y Autores' donde listes los papers consultados.
CADA CITA DEBE SEGUIR EL FORMATO:
- Título del paper.
- Autor(es).
- Año de publicación.
- DOI o enlace directo si está disponible.
"""

agente_cientifico = create_react_agent(model=llm, tools=[tool_cientifica], prompt=prompt_etica_doctrina)

# 3. FUNCIONES DE LOS NODOS
def funcion_agente_web(state: AgentState) -> dict:
    print(f"\n[LOG] >> Agente Web (Tavily) activado para: {state['user_query']}")
    try:
        resultado = agente_web.invoke({"messages": [("user", state["user_query"])]})
        mensaje = resultado["messages"][-1]
        texto_final = mensaje.content[0]['text'] if isinstance(mensaje.content, list) else mensaje.content
        return {"web_answer": texto_final}
    except Exception as e:
        if "429" in str(e):
            return {"web_answer": "⚠️ Lo siento, hemos alcanzado el límite de consultas disponibles (cuota API). Por favor, intenta de nuevo más tarde o mañana."}
        return {"web_answer": f"⚠️ Error inesperado en Web: {str(e)}"}

def funcion_agente_cientifico(state: AgentState) -> dict:
    print(f"\n[LOG] >> Agente Científico (arXiv) activado para: {state['user_query']}")
    try:
        resultado = agente_cientifico.invoke({"messages": [("user", state["user_query"])]})
        mensaje = resultado["messages"][-1]
        texto_final = mensaje.content[0]['text'] if isinstance(mensaje.content, list) else mensaje.content
        return {"scientific_answer": texto_final}
    except Exception as e:
        if "429" in str(e):
            return {"scientific_answer": "⚠️ Lo siento, hemos alcanzado el límite de consultas disponibles en la base científica (cuota API). Por favor, intenta de nuevo más tarde o mañana."}
        return {"scientific_answer": f"⚠️ Error inesperado en Científico: {str(e)}"}

In [ ]:
from langchain_core.messages import HumanMessage

# --- EL ROUTER ---

def router_agent(state: AgentState) -> dict:
    query = state['user_query'].lower()

# Prioridad: garantiza precisión en temas académicos
    if any(word in query for word in ["paper", "academic", "research", "bias", "arxiv"]): #para que sea bien preciso al clasificar
        return {"router_decision": "scientific_search"}

# Clasificación inteligente mediante LLM
    router_prompt = f"Clasifica: '{state['user_query']}'. Responde SÓLO con 'web_search' o 'scientific_search'."
    response = llm.invoke([HumanMessage(content=router_prompt)])

    decision = response.content.strip().replace("'", "").replace(".", "").replace('"', "").lower()

    return {"router_decision": decision}

# Cambio Supervisor 08-Junio-2026 cita agente y fuente
def supervisor_node(state: AgentState) -> dict:
    decision = state.get("router_decision", "No definido")
    web_res = state.get("web_answer", "")
    sci_res = state.get("scientific_answer", "")

    dictamen = "## ⚖️ Dictamen del Auditor Legal y Ético de IA\n"
    dictamen += f"**Agente operativo encargado:** {decision}\n\n"

    # --- Lógica dinámica ---
    if decision == "web_search":
        dictamen += "### 🌐 Informe Legal (Fuente: Tavily):\n"
        dictamen += web_res + "\n\n"
    elif decision == "scientific_search":
        dictamen += "### 🔬 Análisis Científico (Fuente: arXiv):\n"
        dictamen += sci_res + "\n\n"

    dictamen += "\n---\n*Nota: Informe generado con Gemini 2.5 flash.*"
    return {"final_answer": dictamen}

In [ ]:
from langgraph.graph import START, StateGraph, END
from IPython.display import Image, display

workflow = StateGraph(AgentState)

# 1. Mapeo los nodos
workflow.add_node("router", router_agent)
workflow.add_node("agente_investigador_legal", funcion_agente_web)
workflow.add_node("agente_auditor_normativa", funcion_agente_cientifico)
workflow.add_node("supervisor", supervisor_node)

# 2. Punto de entrada
workflow.add_edge(START, "router")

# 3. El router traduce su decisión al nombre real del nodo
workflow.add_conditional_edges(
    "router",
    lambda state: state["router_decision"],
    {
        "web_search": "agente_investigador_legal",
        "scientific_search": "agente_auditor_normativa",
    }
)

# 4. Conexiones de salida
workflow.add_edge("agente_investigador_legal", "supervisor")
workflow.add_edge("agente_auditor_normativa", "supervisor")
workflow.add_edge("supervisor", END)

# 5. Compilación
app = workflow.compile()

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except:
    print("Grafo compilado con éxito.")

In [ ]:
import gradio as gr

def ejecutar_auditoria_web(pregunta_usuario):
    """
    Toma la pregunta de la interfaz de Gradio, hace correr el grafo
    y extrae la respuesta final formateada por el supervisor.
    """
    try:
        # Invoco el grafo con la estructura de estado que definí
        resultado_proceso = app.invoke({"user_query": pregunta_usuario})
        return resultado_proceso["final_answer"]
    except Exception as e:
        return f"⚠️ Ocurrió un error en la ejecución del flujo: {str(e)}"

# Configuro la pantalla del chat
interfaz_chat = gr.Interface(
    fn=ejecutar_auditoria_web,
    inputs=gr.Textbox(label="Escribe tu consulta legal o ética sobre IA:"),
    outputs=gr.Markdown(label="Dictamen Final del Supervisor"),
    title="⚖️ Auditor Legal Inteligente",
    description="Sistema Multi-Agente enfocado en Derecho Digital y Ética, impulsado por LangGraph y Gemini 2.5 Flash", # ⬅️ ¡La coma va aquí, después de las comillas!

    # CAMBIO EL IDIOMA DE LOS BOTONES
    submit_btn="Buscar",    # Cambia "Submit" por "Buscar"
    clear_btn="Limpiar",    # Cambia "Clear" por "Limpiar"
    allow_flagging="never"  # Quita el botón de "Flag" que aparece en inglés
)

# Lanzo con share=True para que me genere el link público
interfaz_chat.launch(share=True, debug=True)